In [1]:
from mylib.statistic_test import *

code_id = "0878 - R4 Length"
loc = join(figpath, "Dsp", code_id)
os.makedirs(loc, exist_ok=True)

d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
if os.path.exists(join(figdata, f"{code_id}.pkl")) == False:
    Length = {
        "MiceID": [],
        "Session": [],
        "Length": []
    }
    for mouse in [10212, 10224, 10227, 10232]:
        file_idx = np.where(f2['MiceID'] == mouse)[0]
        length = []
        for s in tqdm(range(7)):
            with open(f2['Trace File'][file_idx[s]], 'rb') as f:
                trace = pickle.load(f)

            beg, end = LapSplit(trace, trace['paradigm'])
            route = classify_lap(spike_nodes_transform(trace['correct_nodes'], 12), beg, 1)
            for i, b, e in zip(range(len(beg)), beg, end):
                if route[i] != 3:
                    continue
                dx = trace['correct_pos'][b:e, 0]
                dy = trace['correct_pos'][b:e, 1]
                length.append(np.sum(np.sqrt(np.diff(dx)**2 + np.diff(dy)**2)))
            del trace
        
        length = np.median(np.asarray(length))
        Length["MiceID"].append(mouse)
        Length["Session"].append(s)
        Length["Length"].append(length)

    for k in Length.keys():
        Length[k] = np.array(Length[k])
    with open(join(figdata, f"{code_id}.pkl"), 'wb') as f:
        pickle.dump(Length, f)

    LengthD = pd.DataFrame(Length)
    LengthD.to_excel(join(figdata, f"{code_id}.xlsx"), index=False)
else:
    with open(join(figdata, f"{code_id}.pkl"), 'rb') as f:
        Length = pickle.load(f)

print_estimator(Length['Length'])

  Mean: 1565.4586454162536, STD: 107.42045738065882, Max: 1748.856045348704, Min: 1475.311808031545, Median: 1518.8333641423824, df: 3
